### Data Ingestion


In [2]:
### Document Structure

from langchain_core.documents import Document


In [3]:
import os
import uuid
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# 1. Structural Hierarchy for your Textbook Markdown
headers_to_split_on = [
    ("#", "header1"),
    ("##", "header2"), 
    ("###", "header3"),
    ("####", "header4")
]

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False # Keep headers in text so LLM understands the context
)

# 2. Atomic Refinement (The "Child" Splitter)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, 
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def process_markdown_file(file_path, default_subject, folder_type):
    """Processes a single MD file and returns filtered atomic chunks."""
    chunks_from_file = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return []
    
    # Step A: Structural Split (Parent Context)
    header_splits = header_splitter.split_text(content)
    
    # Step B: Atomic Split (Child Chunks)
    refined_splits = text_splitter.split_documents(header_splits)
    
    # Unique ID per file to group related chunks
    parent_id = str(uuid.uuid4())[:8]

    for i, chunk in enumerate(refined_splits):
        # --- NEW: Content Length Filter ---
        # If the chunk is just a header or empty (less than 50 chars), skip it
        if len(chunk.page_content.strip()) < 50:
            continue

        # Step C: Dynamic Subject Tagging (for Mixed PCM files)
        header_context = str(chunk.metadata.values()).lower()
        assigned_subject = default_subject
        
        if any(x in header_context for x in ["physics", "phy"]):
            assigned_subject = "phy"
        elif any(x in header_context for x in ["chemistry", "chem"]):
            assigned_subject = "chem"
        elif any(x in header_context for x in ["math", "maths"]):
            assigned_subject = "maths"

        # Step D: Metadata Enrichment
        chunk.metadata.update({
            "subject": assigned_subject,
            "type": folder_type,
            "source": os.path.basename(file_path),
            "parent_id": parent_id,
            "chunk_index": i,
            "is_example": "example" in header_context
        })
        
        chunks_from_file.append(chunk)
        
    return chunks_from_file

def ingest_all_jee_data(root_path_str="../data"):
    all_chunks = []
    base_dir = Path(root_path_str).resolve() 
    
    print(f"🔍 Starting Ingestion from: {base_dir}")
    subjects = ['physics', 'mathematics', 'chemistry']

    # Part 1: Subject-Specific Textbooks
    for sub in subjects:
        # Looking for 'textbook' or 'textbooks'
        sub_dir = base_dir / sub
        target_folders = list(sub_dir.glob("textbooks"))
        
        for folder in target_folders:
            print(f"📂 Scanning: {folder}")
            for file_path in folder.glob("*.md"):
                all_chunks.extend(process_markdown_file(str(file_path), sub, 'textbooks'))

    # Part 2: Mixed PCM PYQ Folder
    mixed_path = base_dir / "mixed" / "pyq"
    if mixed_path.exists():
        print(f"📂 Scanning Mixed Folder: {mixed_path}")
        for file_path in mixed_path.glob("*.md"):
            all_chunks.extend(process_markdown_file(str(file_path), "mixed", "pyq"))

    return all_chunks

# --- EXECUTION ---
if __name__ == "__main__":
    chunks = ingest_all_jee_data()
    print(f"\n🚀 Successfully generated {len(chunks)} high-quality chunks.")
    
    # Verification: Print a sample chunk to check quality
    if chunks:
        print("\n--- SAMPLE CHUNK VERIFICATION ---")
        sample = chunks[65]
        print(f"Content: {sample.page_content[:150]}...")
        print(f"Metadata: {sample.metadata}")

d:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔍 Starting Ingestion from: D:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\data
📂 Scanning: D:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\data\physics\textbooks
📂 Scanning: D:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\data\mathematics\textbooks
📂 Scanning: D:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\data\chemistry\textbooks

🚀 Successfully generated 13827 high-quality chunks.

--- SAMPLE CHUNK VERIFICATION ---
Content: . Similarly, the then accepted wave picture of light failed to explain the photoelectric effect properly. This led to the development of a radically n...
Metadata: {'header1': 'Appendices', 'header2': 'The Method of Science', 'subject': 'physics', 'type': 'textbooks', 'source': 'Physics_NCERT_Class_11.md', 'parent_id': 'ce7de9fd', 'chunk_index': 107, 'is_example': False}


In [4]:
print(chunks[989])

page_content='A simpler version of the right hand rule is the following : Open up your right hand palm and curl the fingers pointing from $\mathbf{a}$ to $\mathbf{b}$. Your stretched thumb points in the direction of $\mathbf{c}$.  
It should be remembered that there are two angles between any two vectors $\mathbf{a}$ and $\mathbf{b}$. In Fig. 7.15 (a) or (b) they correspond to $\theta$ (as shown) and $(360^{\circ}- \theta)$. While applying either of the above rules, the rotation should be taken through the smaller angle ($<180^{\circ}$) between $\mathbf{a}$ and $\mathbf{b}$. It is $\theta$ here.  
Because of the cross ($\times$) used to denote the vector product, it is also referred to as cross product.  
*   Note that scalar product of two vectors is commutative as said earlier, $\mathbf{a}.\mathbf{b} = \mathbf{b}.\mathbf{a}$
*   The vector product, however, is not commutative, i.e. $\mathbf{a} \times \mathbf{b} \neq \mathbf{b} \times \mathbf{a}$' metadata={'header1': 'Work, Energy an

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

# 1. Choose a high-quality free embedding model
# 'all-MiniLM-L6-v2' is fast and great for general purposes
# 'BAAI/bge-small-en-v1.5' is one of the best for high accuracy
model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {'device': 'cpu'} # Use 'cuda' if you have an NVIDIA GPU
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# 2. Define where you want to save your data
CHROMA_PATH = "./chroma_db"

# 3. Create and store the embeddings
# This command converts text to vectors and saves them to CHROMA_PATH
print(f"⏳ Encoding {len(chunks)} chunks... This may take a moment.")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
    collection_name="jee_knowledge_base"
)

print(f"✅ Successfully stored {len(chunks)} chunks in {CHROMA_PATH}")

d:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Load the same model and path
CHROMA_PATH = "./chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 2. Connect to the existing DB
db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings, collection_name="jee_knowledge_base")

# 3. Perform a test search
query = "what is the difference between speed and velocity"
docs = db.similarity_search(query, k=3)

print(f"\n🎯 Top Results for: {query}\n" + "="*50)
for i, doc in enumerate(docs):
    print(f"Result {i+1} | Source: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content[:300]}...\n")

d:\Projects and folders\FInal_Projects\JEE_Saathi_AI\JEE_Saathi_AI\Folder\RAG\JEESaathiAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🎯 Top Results for: what is the difference between speed and velocity
Result 1 | Source: Physics_NCERT_Class_11.md
Content: ### 3.15
In Exercises 3.13 and 3.14, we have carefully distinguished between average speed and magnitude of average velocity.
We consider instantaneous speed and magnitude of instantaneous velocity.
The instantaneous speed is always equal to the magnitude of instantaneous velocity.
Why ?...

Result 2 | Source: Physics_Part_2_Class_11.md
Content: 7. For an observer moving with velocity $v_{o}$ relative to the medium, the speed of a wave is obviously different from $v$ and is given by $v \pm v_{o}$. [No diagram present on this page]  
--- Page 171 ---...

Result 3 | Source: Physics_NCERT_Class_11.md
Content: ### 3.13
Explain clearly, with examples, the distinction between :
(a) magnitude of displacement (sometimes called distance) over an interval of time,
and the total length of path covered by a particle over the same interval;  
(b) magnitude of average velocity 